## Estruturação de um arquivo CSV com a média histórica e a tendência do índice IDEB de cada município brasileiro, para uso no treinamento do modelo.
####Imputação feita utilizando o IDEB do estado.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
city_code = pd.read_csv("/content/br_bd_diretorios_brasil_municipio.csv", delimiter=';')
city_ideb = pd.read_csv("/content/br_ideb_municipio.csv", delimiter=';')

In [ ]:
city_code = city_code.rename(columns={"nome": "nome_municipio"})
city_code.head()

,id_municipio,nome_municipio,sigla_uf
0,5101837,Boa Esperança do Norte,MT
1,1100809,Candeias do Jamari,RO
2,1100338,Nova Mamoré,RO
3,1100205,Porto Velho,RO
4,1101104,Itapuã do Oeste,RO


In [ ]:
city_ideb.head()

,sigla_uf,id_municipio,ano,ideb
0,AC,1200013,2017,5.40
1,AC,1200013,2019,5.60
2,AC,1200013,2021,5.20
3,AC,1200054,2017,4.60
4,AC,1200054,2019,4.95


In [ ]:
score_2017 = city_ideb[city_ideb['ano'] == 2017].set_index("id_municipio")["ideb"]
score_2021 = city_ideb[city_ideb['ano'] == 2021].set_index("id_municipio")["ideb"]

tendencia = score_2021 - score_2017

city_ideb["tendencia_ideb"] = city_ideb['id_municipio'].map(tendencia)

In [ ]:
city_ideb = city_ideb.groupby(by=["sigla_uf", "id_municipio", "tendencia_ideb"]).agg(ideb_medio = ("ideb", "mean")).reset_index()
city_ideb.head()

,sigla_uf,id_municipio,tendencia_ideb,ideb_medio
0,AC,1200013,-0.200000,5.400000
1,AC,1200054,0.050000,4.733333
2,AC,1200104,-0.633333,6.633333
3,AC,1200138,0.950000,4.866667
4,AC,1200179,1.300000,4.766667


In [ ]:
city_ideb['id_municipio'].value_counts().sum()

np.int64(5567)

In [ ]:
city_code['id_municipio'].value_counts().sum()

np.int64(5571)

In [ ]:
final_data = city_ideb.merge(city_code, how="left")
final_data

,sigla_uf,id_municipio,tendencia_ideb,ideb_medio,nome_municipio
0,AC,1200013,-0.200000,5.400000,Acrelândia
1,AC,1200054,0.050000,4.733333,Assis Brasil
2,AC,1200104,-0.633333,6.633333,Brasiléia
3,AC,1200138,0.950000,4.866667,Bujari
4,AC,1200179,1.300000,4.766667,Capixaba
...,...,...,...,...,...
5562,TO,1721208,-0.450000,5.061111,Tocantinópolis
5563,TO,1721257,0.600000,5.400000,Tupirama
5564,TO,1721307,0.500000,4.866667,Tupiratins
5565,TO,1722081,0.133333,4.722222,Wanderlândia


In [ ]:
final_data = final_data.round({"tendencia_ideb": 2, "ideb_medio": 2})
final_data

,sigla_uf,id_municipio,tendencia_ideb,ideb_medio,nome_municipio
0,AC,1200013,-0.20,5.40,Acrelândia
1,AC,1200054,0.05,4.73,Assis Brasil
2,AC,1200104,-0.63,6.63,Brasiléia
3,AC,1200138,0.95,4.87,Bujari
4,AC,1200179,1.30,4.77,Capixaba
...,...,...,...,...,...
5562,TO,1721208,-0.45,5.06,Tocantinópolis
5563,TO,1721257,0.60,5.40,Tupirama
5564,TO,1721307,0.50,4.87,Tupiratins
5565,TO,1722081,0.13,4.72,Wanderlândia


In [ ]:
final_data.to_csv("/content/br_historico_ideb_municipal.csv")